In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [3]:
from src.data_loader import load_config, load_cell2cell_train, split_data
from src.feature_engineering import Cell2CellFeatureEngineer

In [4]:
config = load_config()

X, y = load_cell2cell_train(config)

print("Original Shape :", X.shape)

2026-06-26 18:41:04.915 | INFO     | src.data_loader:load_cell2cell_train:27 - Loading Cell2Cell training data...
2026-06-26 18:41:05.468 | INFO     | src.data_loader:load_cell2cell_train:33 - Loaded: (51047, 58)
2026-06-26 18:41:05.478 | INFO     | src.data_loader:load_cell2cell_train:42 - Churn rate: 0.288 (14711 / 51047 churners)
2026-06-26 18:41:05.503 | INFO     | src.data_loader:load_cell2cell_train:63 - After drops: (51047, 52)


Original Shape : (51047, 52)


In [5]:
X_train, X_test, y_train, y_test = split_data(X, y, config)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)


2026-06-26 18:41:22.431 | INFO     | src.data_loader:split_data:260 - Train: (40837, 52) | Churn rate: 0.288
2026-06-26 18:41:22.431 | INFO     | src.data_loader:split_data:265 - Test: (10210, 52) | Churn rate: 0.288


Train : (40837, 52)
Test  : (10210, 52)


In [6]:
fe = Cell2CellFeatureEngineer()

X_train_fe = fe.fit_transform(X_train, y_train)
X_test_fe = fe.transform(X_test)

2026-06-26 18:41:40.920 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded ServiceArea: 729 levels
2026-06-26 18:41:40.930 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded Occupation: 8 levels
2026-06-26 18:41:40.939 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded PrizmCode: 4 levels
2026-06-26 18:41:40.950 | DEBUG    | src.feature_engineering:fit:47 - Target-encoded CreditRating: 7 levels
2026-06-26 18:41:41.091 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (40837, 85)
2026-06-26 18:41:41.122 | DEBUG    | src.feature_engineering:transform:218 - FE complete. Shape: (10210, 85)


In [7]:
print("\nOriginal Columns :", X_train.shape[1])
print("New Columns      :", X_train_fe.shape[1])

print("\nRows unchanged :", X_train.shape[0] == X_train_fe.shape[0])

print("\nescalated_customer exists:",
      "escalated_customer" in X_train_fe.columns)

if "escalated_customer" in X_train_fe.columns:
    print("Sum:", X_train_fe["escalated_customer"].sum())

print("\nretention_failed exists:",
      "retention_failed" in X_train_fe.columns)

print("\nservice_area_encoded exists:",
      "service_area_encoded" in X_train_fe.columns)

print("\ntenure_group dtype:")

if "tenure_group" in X_train_fe.columns:
    print(X_train_fe["tenure_group"].dtype)

print("\ncredit_risk_score dtype:")

if "credit_risk_score" in X_train_fe.columns:
    print(X_train_fe["credit_risk_score"].dtype)
    print(
        "Min:",
        X_train_fe["credit_risk_score"].min(),
        "Max:",
        X_train_fe["credit_risk_score"].max()
    )

print("\nlog_care_calls exists:",
      "log_care_calls" in X_train_fe.columns)



Original Columns : 52
New Columns      : 85

Rows unchanged : True

escalated_customer exists: True
Sum: 323

retention_failed exists: True

service_area_encoded exists: True

tenure_group dtype:
category

credit_risk_score dtype:
int64
Min: 1 Max: 7

log_care_calls exists: True


In [8]:
new_features = sorted(
    list(set(X_train_fe.columns) - set(X_train.columns))
)

for col in new_features:
    print("-", col)

print("\nTotal New Features:", len(new_features))

- credit_risk_score
- device_diversity
- dropped_rate
- early_tenure
- equipment_age_months
- escalated_customer
- handset_tier
- has_overage
- inactive_subs_ratio
- is_family_plan
- is_low_credit
- log_care_calls
- log_equipment_days
- log_overage
- loyal_customer
- occupation_encoded
- old_equipment
- peak_ratio
- premium_dissatisfied
- premium_handset
- prizm_encoded
- retention_failed
- retention_success_rate
- revenue_declining
- revenue_growing
- revenue_per_minute
- service_area_encoded
- tenure_group
- total_call_volume
- total_complaint_calls
- usage_declining
- usage_declining_severe
- very_old_equipment

Total New Features: 33


In [9]:
temp = X_train_fe.copy()
temp["Churn"] = y_train.values

print(temp.groupby("retention_failed")["Churn"].mean())

retention_failed
0    0.284985
1    0.474674
Name: Churn, dtype: float64
